# CEREBRO PoC — 02G Submit & Artifact Registration

**Experiment:** EXP-INGEST-006  
**Stage:** Artifact Registration  
**Purpose:** Commit a human-approved staged artifact into CEREBRO.

---

## Objective

Validate that only a human-reviewed and explicitly approved artifact can be registered into the Digital Knowledge Twin.

Registration creates a stable artifact identity while preserving the complete provenance chain:

**Original Source → Deterministic Extraction → AI Enrichment → Human Review → Registered Artifact**

Registration does not yet create knowledge fragments. It establishes the trusted source artifact from which knowledge may subsequently be constructed.

### 1 — Load 02F approved result

In [3]:
from pathlib import Path
import json
import hashlib

repo_root = Path.cwd().parents[1]

approval_path = (
    repo_root
    / "poc/data/processed/EXP-INGEST-005-approved.json"
)

assert approval_path.exists(), (
    f"Approved 02F output not found: {approval_path}"
)

with open(
    approval_path,
    "r",
    encoding="utf-8"
) as f:
    approved_review = json.load(f)

assert approved_review["status"] == "approved"
assert approved_review["user_approval"] is True

print("✓ Human-approved artifact loaded")
print("✓ Registration gate passed")

✓ Human-approved artifact loaded
✓ Registration gate passed


### 2 — Resolve and verify original source

In [4]:
source_path = (
    repo_root
    / "poc/data/raw/text/benchmark_001.txt"
)

assert source_path.exists()

file_bytes = source_path.read_bytes()

source_sha256 = hashlib.sha256(
    file_bytes
).hexdigest()

print("✓ Original source resolved")
print("Source :", source_path.name)
print("SHA256 :", source_sha256)

✓ Original source resolved
Source : benchmark_001.txt
SHA256 : 7694567acb26611218fd818ef2c12ec456e5ee94c5387804bfe5f36799aee832


### 3 — Register artifact

In [5]:
fields = approved_review["fields"]

registered_artifact = {
    "artifact_id": "ART-0001",

    "status": "registered",

    "source": {
        "filename": source_path.name,
        "storage_uri": "poc/data/raw/text/benchmark_001.txt",
        "sha256": source_sha256
    },

    "metadata": {
        field: data["value"]
        for field, data in fields.items()
    },

    "ownership": {
        "owner": "artifact_owner"
    },

    "provenance": {
        "deterministic_extraction": "EXP-INGEST-002",
        "ai_enrichment": "EXP-INGEST-004",
        "human_review": "EXP-INGEST-005",
        "registration": "EXP-INGEST-006"
    },

    "integrity": {
        "human_reviewed": True,
        "human_approved": True,
        "source_verified": True
    }
}

registered_artifact

{'artifact_id': 'ART-0001',
 'status': 'registered',
 'source': {'filename': 'benchmark_001.txt',
  'storage_uri': 'poc/data/raw/text/benchmark_001.txt',
  'sha256': '7694567acb26611218fd818ef2c12ec456e5ee94c5387804bfe5f36799aee832'},
 'metadata': {'title': 'CEREBRO Digital Knowledge Twin',
  'artifact_type': 'knowledge_description',
  'language': 'English',
  'description': 'CEREBRO is a Digital Knowledge Twin designed to preserve and connect human knowledge while maintaining provenance to supporting source artifacts.',
  'authors': [],
  'people': [],
  'organizations': [],
  'projects': ['CEREBRO'],
  'topics': ['Digital Knowledge Twin',
   'Provenance',
   'Knowledge Fragments',
   'Source Artifacts',
   'Assisted Recollection'],
  'tags': ['CEREBRO', 'knowledge', 'provenance', 'recollection']},
 'ownership': {'owner': 'artifact_owner'},
 'provenance': {'deterministic_extraction': 'EXP-INGEST-002',
  'ai_enrichment': 'EXP-INGEST-004',
  'human_review': 'EXP-INGEST-005',
  'registra

### 4 — Integrity gate

In [8]:
assert registered_artifact["status"] == "registered"

assert registered_artifact[
    "integrity"
]["human_reviewed"]

assert registered_artifact[
    "integrity"
]["human_approved"]

assert registered_artifact[
    "integrity"
]["source_verified"]

assert (
    registered_artifact["source"]["sha256"]
    == source_sha256
)

print("✓ Human review verified")
print("✓ Human approval verified")
print("✓ Original source verified")
print("✓ Provenance chain retained")

✓ Human review verified
✓ Human approval verified
✓ Original source verified
✓ Provenance chain retained


### 5 — Persist registered artifact

In [9]:
registry_dir = (
    repo_root
    / "poc/data/processed/artifacts"
)

registry_dir.mkdir(
    parents=True,
    exist_ok=True
)

artifact_path = (
    registry_dir
    / "ART-0001.json"
)

with open(
    artifact_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        registered_artifact,
        f,
        indent=2,
        ensure_ascii=False
    )

print("✓ Artifact registered")
print("Artifact ID:", registered_artifact["artifact_id"])
print("Registry:", artifact_path)

✓ Artifact registered
Artifact ID: ART-0001
Registry: /Users/joeldizon/development/cerebro_dev/cerebro/poc/data/processed/artifacts/ART-0001.json


### 6 — Final result

In [10]:
print("CEREBRO — Artifact Registration")
print("=" * 50)

print("Artifact :", registered_artifact["artifact_id"])
print("Status   :", registered_artifact["status"])
print("Source   :", registered_artifact["source"]["filename"])
print("Reviewed :", registered_artifact["integrity"]["human_reviewed"])
print("Approved :", registered_artifact["integrity"]["human_approved"])
print("Verified :", registered_artifact["integrity"]["source_verified"])

print("\n✓ EXP-INGEST-006: PASS")

CEREBRO — Artifact Registration
Artifact : ART-0001
Status   : registered
Source   : benchmark_001.txt
Reviewed : True
Approved : True
Verified : True

✓ EXP-INGEST-006: PASS
